# CLV Validation: Temporal Holdout Backtesting

This notebook validates the BG/NBD + Gamma-Gamma CLV model against a **180-day holdout window** (2025-04-04 → 2025-10-01).

**Validation strategy:**
1. Scale 12-month predictions to 180-day holdout window (× 180/365)
2. Transaction-level accuracy: Pearson r + MAE vs `actual_holdout_transactions`
3. Revenue-level accuracy: Pearson r + MAE + MAPE vs `actual_holdout_revenue`
4. **Lift curve**: does the model correctly rank customers by value?
5. Goodness-of-fit: `plot_period_transactions` diagnostic

**Target:** Top 20% by predicted CLV should capture ≥50% of actual holdout revenue.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
import warnings
warnings.filterwarnings('ignore')

from lifetimes.plotting import plot_period_transactions
from scipy.stats import pearsonr

# Paths
SCORED_PATH = "../data/processed/clv_scored.csv"
BGNBD_PATH  = "../models/bgnbd_model.pkl"

# Holdout window
HOLDOUT_DAYS = 180
CLV_DAYS     = 365
HOLDOUT_SCALE = HOLDOUT_DAYS / CLV_DAYS   # ~0.493

print(f"Holdout scale factor: {HOLDOUT_SCALE:.4f} ({HOLDOUT_DAYS}/{CLV_DAYS} days)")

## 1. Load Data and Scale Predictions

In [ ]:
df = pd.read_csv(SCORED_PATH)
bgf = joblib.load(BGNBD_PATH)

print(f"Loaded {len(df):,} customers")

# Scale 12m predictions to holdout window
df['predicted_purchases_holdout'] = df['predicted_purchases_12m'] * HOLDOUT_SCALE
df['predicted_revenue_holdout']   = df['clv_12m'] * HOLDOUT_SCALE

# Holdout buyers only (for revenue MAPE)
holdout_buyers = df[df['actual_holdout_transactions'] > 0].copy()
print(f"\nHoldout buyers: {len(holdout_buyers):,} ({len(holdout_buyers)/len(df):.1%} of customers)")
print(f"Total actual holdout revenue: ${df['actual_holdout_revenue'].sum():,.2f}")

## 2. Transaction-Level Validation

In [ ]:
# Pearson correlation and MAE on transaction counts
r_txn, p_txn = pearsonr(df['actual_holdout_transactions'], df['predicted_purchases_holdout'])
mae_txn = (df['actual_holdout_transactions'] - df['predicted_purchases_holdout']).abs().mean()

print("=== Transaction-Level Metrics ===")
print(f"Pearson r:  {r_txn:.4f}  (p={p_txn:.4f})")
print(f"MAE:        {mae_txn:.4f} transactions per customer")

# Scatter: predicted vs actual transactions
fig, ax = plt.subplots(figsize=(7, 5))
sample = df.sample(min(3000, len(df)), random_state=42)
ax.scatter(sample['predicted_purchases_holdout'], sample['actual_holdout_transactions'],
           alpha=0.2, s=8, color='steelblue')
ax.set_xlabel('Predicted Transactions (holdout-scaled)')
ax.set_ylabel('Actual Transactions (holdout)')
ax.set_title(f'Transaction Prediction Accuracy\n(Pearson r={r_txn:.3f}, MAE={mae_txn:.3f})')
plt.tight_layout()
plt.show()

## 3. Revenue-Level Validation

In [ ]:
# Revenue metrics (all customers)
r_rev, p_rev = pearsonr(df['actual_holdout_revenue'], df['predicted_revenue_holdout'])
mae_rev = (df['actual_holdout_revenue'] - df['predicted_revenue_holdout']).abs().mean()

# MAPE only for holdout buyers (avoid division by zero)
mape_rev = (
    (holdout_buyers['actual_holdout_revenue'] - holdout_buyers['predicted_revenue_holdout']).abs()
    / holdout_buyers['actual_holdout_revenue']
).mean()

print("=== Revenue-Level Metrics ===")
print(f"Pearson r:  {r_rev:.4f}  (p={p_rev:.4f})")
print(f"MAE:        ${mae_rev:.2f} per customer")
print(f"MAPE:       {mape_rev:.1%} (holdout buyers only)")

## 4. Lift Curve (Key Business Metric)

The lift curve answers: *"If we target the top X% by predicted CLV, what % of actual revenue do we capture?"*

**Target:** Top 20% by predicted CLV → ≥50% of actual holdout revenue.

In [ ]:
# Rank customers by predicted CLV (descending)
df_ranked = df.sort_values('clv_12m', ascending=False).reset_index(drop=True)
total_actual_revenue = df_ranked['actual_holdout_revenue'].sum()

df_ranked['cum_actual_revenue'] = df_ranked['actual_holdout_revenue'].cumsum()
df_ranked['cum_revenue_pct']    = df_ranked['cum_actual_revenue'] / total_actual_revenue * 100
df_ranked['customer_pct']       = (df_ranked.index + 1) / len(df_ranked) * 100

# Random baseline (diagonal)
baseline = df_ranked['customer_pct'].values

# Compute top-20% capture rate
top20_idx = int(0.20 * len(df_ranked))
top20_revenue_pct = df_ranked.iloc[:top20_idx]['actual_holdout_revenue'].sum() / total_actual_revenue
print(f"Top 20% by predicted CLV captures {top20_revenue_pct:.1%} of actual holdout revenue")
if top20_revenue_pct >= 0.50:
    print("✓ Lift target met: ≥50% revenue captured in top 20%")
else:
    print(f"⚠ Lift target not met (target: ≥50%, actual: {top20_revenue_pct:.1%})")

# Plot lift curve
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(df_ranked['customer_pct'], df_ranked['cum_revenue_pct'],
        color='steelblue', linewidth=2.5, label='BG/NBD + Gamma-Gamma CLV')
ax.plot(baseline, baseline, color='gray', linestyle='--', linewidth=1.5, label='Random baseline')
ax.axvline(20, color='red', linestyle=':', alpha=0.8)
ax.axhline(top20_revenue_pct * 100, color='red', linestyle=':', alpha=0.8)
ax.annotate(
    f'Top 20% → {top20_revenue_pct:.1%} of revenue',
    xy=(20, top20_revenue_pct * 100),
    xytext=(30, top20_revenue_pct * 100 - 10),
    fontsize=10,
    arrowprops=dict(arrowstyle='->', color='red'),
    color='red'
)
ax.set_xlabel('% of Customers (ranked by predicted CLV)')
ax.set_ylabel('% of Actual Holdout Revenue Captured')
ax.set_title('Lift Curve: Revenue Capture by Predicted CLV Rank')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. BG/NBD Goodness-of-Fit

`plot_period_transactions` overlays the actual transaction frequency distribution against the model's expected distribution. Bars should align closely.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_period_transactions(bgf, ax=ax, max_frequency=10)
ax.set_title('BG/NBD Goodness-of-Fit: Actual vs Expected Transaction Frequency')
plt.tight_layout()
plt.show()

## 6. Validation Summary

In [ ]:
print("=" * 55)
print("CLV MODEL VALIDATION SUMMARY")
print("=" * 55)
print(f"\nHoldout window:       {HOLDOUT_DAYS} days")
print(f"Total customers:      {len(df):,}")
print(f"Holdout buyers:       {len(holdout_buyers):,} ({len(holdout_buyers)/len(df):.1%})")
print(f"\nTransaction accuracy:")
print(f"  Pearson r:          {r_txn:.4f}")
print(f"  MAE:                {mae_txn:.4f} transactions")
print(f"\nRevenue accuracy:")
print(f"  Pearson r:          {r_rev:.4f}")
print(f"  MAE:                ${mae_rev:.2f} per customer")
print(f"  MAPE (buyers only): {mape_rev:.1%}")
print(f"\nLift curve:")
print(f"  Top 20% CLV →       {top20_revenue_pct:.1%} of holdout revenue")
print(f"  Target met?         {'✓ YES' if top20_revenue_pct >= 0.50 else '✗ NO (target: 50%)'}")
print("=" * 55)

---

**Next:** [04_clv_segmentation.ipynb](./04_clv_segmentation.ipynb) — 4-tier customer segmentation, XGBoost CLV regressor, and campaign ROI allocation